In [ ]:
%pip install ollama nest_asyncio

import ollama
import asyncio
import nest_asyncio
from typing import List, Dict
import json

# Patch pour permettre l'async dans Jupyter
nest_asyncio.apply()

MODEL_NAME = "llama3.2" 

print(f"Configuration chargée. Modèle utilisé : {MODEL_NAME}")

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
     -------------------------------------- 463.6/463.6 kB 7.3 MB/s eta 0:00:00
     -------------------------------------- 113.4/113.4 kB 6.4 MB/s eta 0:00:00
     -------------------------------------- 159.4/159.4 kB 9.3 MB/s eta 0:00:00
  Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
     ---------------------------------------- 71.0/71.0 kB ? eta 0:00:00
  Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
     ---------------------------------------- 2.0/2.0 MB 21.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Configuration chargée. Modèle utilisé : llama3.2


In [3]:
def prompt_chaining_workflow(text_input):
    print(f"--- Démarrage du Chaining pour : '{text_input[:30]}...' ---")
    
    # Étape 1 : Résumer
    print("1. Résumé en cours...")
    res1 = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'user', 'content': f"Résume le texte suivant en une seule phrase concise : {text_input}"}
    ])
    summary = res1['message']['content']
    print(f"   -> {summary}")

    # Étape 2 : Traduire
    print("2. Traduction en anglais...")
    res2 = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'user', 'content': f"Translate this french text to English: {summary}"}
    ])
    translation = res2['message']['content']
    print(f"   -> {translation}")

    # Étape 3 : Reformuler (Style formel)
    print("3. Reformulation formelle...")
    res3 = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'user', 'content': f"Reformulate this text to be extremely formal and professional: {translation}"}
    ])
    formal = res3['message']['content']
    print(f"   -> {formal}")

    # Étape 4 : Simplifier (Explain like I'm 5)
    print("4. Simplification...")
    res4 = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'user', 'content': f"Explain this text simply for a 5 year old child: {formal}"}
    ])
    final = res4['message']['content']
    
    print("\n=== RÉSULTAT FINAL ===")
    print(final)
    return final

# Test
texte_source = "L'intelligence artificielle générative permet de créer du contenu nouveau, comme du texte, des images ou du code, en apprenant des modèles à partir de vastes bases de données existantes."
prompt_chaining_workflow(texte_source)

--- Démarrage du Chaining pour : 'L'intelligence artificielle gé...' ---
1. Résumé en cours...
   -> L'intelligence artificielle générative est un domaine qui utilise l'apprentissage automatique pour créer du contenu original à partir de grandes quantités de données existantes.
2. Traduction en anglais...
   -> The translation of the French text is:

"Artificial intelligence generative is a field that uses machine learning to create original content from large quantities of existing data."

Here's a breakdown of the translation:

* "L'intelligence artificielle générative" = Artificial Intelligence Generative
* "est un domaine" = is a field or domain
* "qui utilise l'apprentissage automatique" = that uses machine learning
* "pour créer du contenu original à partir de grandes quantités de données existantes" = to create original content from large quantities of existing data

Let me know if you have any further requests!
3. Reformulation formelle...
   -> Here is the reformulated text in

'Here\'s an explanation of the text in simple words for a 5-year-old:\n\nThere\'s something called "Artificial Intelligence Generative". It means that computers can make new things, like pictures or stories, using lots and lots of old things they\'ve seen before.\n\nThink of it like this: Imagine you have a toy box full of blocks in different shapes and colors. A computer with Artificial Intelligence Generative can look at all those blocks and create new blocks on its own, just like the ones you want to see!\n\nBut how does it do that? Well, computers are really good at learning from things they see, kind of like how you learn new words when you read books. So, this computer can learn from lots of old pictures or stories and then make brand new ones!'

In [ ]:
def evaluate_quality(original_context, generated_response):
    """
    Utilise un LLM pour noter une réponse selon 3 critères.
    """
    prompt_eval = f"""
    Tu es un juge impartial. Évalue la réponse générée par rapport au contexte original.
    Contexte original : "{original_context}"
    Réponse générée : "{generated_response}"
    
    Donne une note sur 10 pour chacun de ces critères :
    1. Cohérence (Logique de la réponse)
    2. Lisibilité (Clarté du langage)
    3. Fidélité (Respect du sens original)
    
    Réponds UNIQUEMENT au format JSON comme ceci :
    {{
        "coherence": 8,
        "lisibilite": 9,
        "fidelite": 7
    }}
    """
    
    response = ollama.chat(model=MODEL_NAME, messages=[{'role': 'user', 'content': prompt_eval}])
    content = response['message']['content']
    
    # Petit hack pour extraire le JSON si le modèle bavarde autour
    try:
        start = content.find('{')
        end = content.rfind('}') + 1
        json_str = content[start:end]
        scores = json.loads(json_str)
        
        scores['average'] = round(sum(scores.values()) / 3, 2)
        return scores
    except:
        return {"error": "Parsing failed", "raw": content}

# Test
ref = "Le ciel est bleu."
gen = "L'azur céleste arbore une teinte cyan."
print(evaluate_quality(ref, gen))

{'coherence': 6, 'lisibilite': 8, 'fidelite': 2, 'average': 5.33}


In [5]:
def router_agent(user_query):
    print(f"Routing de la demande : '{user_query}'")
    
    prompt = f"""
    Tu es un routeur intelligent. Classifie la demande suivante dans une seule catégorie.
    Catégories possibles :
    - CODE (pour la programmation, python, algorithmes)
    - CREATIVE (pour les poèmes, histoires, idées)
    - SCIENCE (pour les faits, physique, mathématiques)
    - GENERAL (pour le reste)
    
    Demande : "{user_query}"
    
    Réponds UNIQUEMENT par le mot de la catégorie.
    """
    
    res = ollama.chat(model=MODEL_NAME, messages=[{'role': 'user', 'content': prompt}])
    category = res['message']['content'].strip().upper()
    
    # Nettoyage si le modèle met des points
    if "CODE" in category: return "CODE"
    if "CREATIVE" in category: return "CREATIVE"
    if "SCIENCE" in category: return "SCIENCE"
    return "GENERAL"

def handle_request_with_routing(query):
    category = router_agent(query)
    print(f" -> Catégorie détectée : {category}")
    
    system_prompts = {
        "CODE": "Tu es un expert Python senior. Réponds uniquement avec du code optimisé.",
        "CREATIVE": "Tu es un poète romantique. Réponds avec des rimes et du style.",
        "SCIENCE": "Tu es un professeur de physique rigoureux. Sois précis et factuel.",
        "GENERAL": "Tu es un assistant utile."
    }
    
    selected_prompt = system_prompts.get(category, system_prompts["GENERAL"])
    
    response = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'system', 'content': selected_prompt},
        {'role': 'user', 'content': query}
    ])
    return response['message']['content']

# Test
print(handle_request_with_routing("Écris une fonction de tri en Python"))
print("---")
print(handle_request_with_routing("Explique la gravité"))

Routing de la demande : 'Écris une fonction de tri en Python'
 -> Catégorie détectée : CODE
```python
def tri_ascendant(lst):
    """
    Fonction pour trier une liste en ordre ascendant.

    Args:
        lst (list): Liste à trier.

    Returns:
        list: Liste triée en ordre ascendant.
    """
    return sorted(lst)

def tri_descendant(lst):
    """
    Fonction pour trier une liste en ordre descendant.

    Args:
        lst (list): Liste à trier.

    Returns:
        list: Liste triée en ordre descendant.
    """
    return sorted(lst, reverse=True)
```

Exemple d'utilisation :

```python
# Création d'une liste de nombres
lst = [5, 2, 8, 3, 1]

# Tri par ordre ascendant
triAsc = tri_ascendant(lst)
print("Tri Ascendant : ", triAsc)

# Tri par ordre descendant
triDesc = tri_descendant(lst)
print("Tri Descendant : ", triDesc)
```

Remarque : La fonction `sorted()` utilise l'algorithmique de Timsort, qui est une implémentation rapide et efficace du tri. Il est donc recommandé d'u

In [6]:
async def generate_candidate(id, query, system_role):
    """Génère une réponse unique (cette fonction sera lancée en parallèle)"""
    client = ollama.AsyncClient()
    response = await client.chat(model=MODEL_NAME, messages=[
        {'role': 'system', 'content': system_role},
        {'role': 'user', 'content': query}
    ])
    return {
        "id": id,
        "content": response['message']['content']
    }

async def final_boss_workflow(user_query):
    print(f"🚀 DÉMARRAGE DU SUPER-AGENT pour : {user_query}")
    
    # 1. Routing (Optionnel, ici on force la diversité)
    print("--- 1. Génération de 3 candidats en parallèle ---")
    
    # On définit 3 "personnalités" pour avoir des réponses variées
    tasks = [
        generate_candidate(1, user_query, "Tu es concis et direct."),
        generate_candidate(2, user_query, "Tu es très détaillé et pédagogique."),
        generate_candidate(3, user_query, "Tu es critique et analytique.")
    ]
    
    candidates = await asyncio.gather(*tasks)
    
    print("--- 2. Évaluation des candidats ---")
    best_score = -1
    best_response = None
    
    for cand in candidates:
        print(f"\nEvaluation du candidat {cand['id']}...")
        
        scores = evaluate_quality(user_query, cand['content'])
        
        current_score = scores.get('average', 0)
        
        print(f"   -> Score: {current_score}/10 (Cohérence: {scores.get('coherence')}, Lisibilité: {scores.get('lisibilite')})")
        
        if current_score > best_score:
            best_score = current_score
            best_response = cand
            
    print("\n🏆 --- 3. SÉLECTION DU VAINQUEUR --- 🏆")
    print(f"Le meilleur candidat est le n°{best_response['id']} avec un score de {best_score}/10")
    print("\nRéponse finale choisie :")
    print(best_response['content'])

# Lancement du workflow asynchrone
await final_boss_workflow("Explique pourquoi le ciel est bleu")

🚀 DÉMARRAGE DU SUPER-AGENT pour : Explique pourquoi le ciel est bleu
--- 1. Génération de 3 candidats en parallèle ---
--- 2. Évaluation des candidats ---

Evaluation du candidat 1...
   -> Score: 8.33/10 (Cohérence: 9, Lisibilité: None)

Evaluation du candidat 2...
   -> Score: 8.0/10 (Cohérence: 8, Lisibilité: None)

Evaluation du candidat 3...
   -> Score: 8.0/10 (Cohérence: 8, Lisibilité: 9)

🏆 --- 3. SÉLECTION DU VAINQUEUR --- 🏆
Le meilleur candidat est le n°1 avec un score de 8.33/10

Réponse finale choisie :
Le ciel apparentément bleu, mais qu'est-ce qui y fait référence? 

En réalité, rien de permanent ne sera jamais réellement "bleu". Ce que l'on observe comme un bleu s'explique par la diffusion du soleil. Le soleil envoie des photons colorés avec une longueur d'onde de 550 nanomètres et de 570 nanomètres avec une forte proportion de ce qui nous apparaît sous forme de bleu.
